# 网络优化与正则化

本节系统地把 PyTorch 训练时常用的几件工具串一遍：

1. **优化器** —— `torch.optim.{SGD, SGD+momentum, RMSprop, Adam}` 在一个二维损失面上的轨迹对比
2. **mini-batch 大小** 对收敛速度的影响
3. **参数初始化** —— Xavier / Kaiming，对深网络激活方差的稳定作用
4. **BatchNorm** —— 训练曲线对比
5. **Dropout** —— 作为正则化的效果
6. **学习率调度** —— `torch.optim.lr_scheduler`

为了让 notebook 跑得快，每个对比实验用的网络都不大、数据也少。结论方向在更大的网络上更明显。

## 1. 不同优化器在二维 loss 面上的轨迹

经典 Beale 函数：$f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$。
全局最小在 $(3, 0.5)$。从同一起点出发，看不同优化器走到哪里、走得多远。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

def beale(x, y):
    return ((1.5 - x + x*y)**2
            + (2.25 - x + x*y**2)**2
            + (2.625 - x + x*y**3)**2)

def run_optim(opt_factory, n_steps=200):
    xy = torch.tensor([1.0, 1.5], requires_grad=True)
    opt = opt_factory([xy])
    traj = [xy.detach().clone().tolist()]
    for _ in range(n_steps):
        opt.zero_grad()
        beale(xy[0], xy[1]).backward()
        opt.step()
        traj.append(xy.detach().clone().tolist())
    return torch.tensor(traj)

optimizers = {
    'SGD lr=1e-3':         lambda p: optim.SGD(p, lr=1e-3),
    'SGD+momentum=0.9':    lambda p: optim.SGD(p, lr=1e-3, momentum=0.9),
    'RMSprop lr=1e-2':     lambda p: optim.RMSprop(p, lr=1e-2),
    'Adam lr=5e-2':        lambda p: optim.Adam(p, lr=5e-2),
}
trajs = {name: run_optim(fn) for name, fn in optimizers.items()}

# 绘制 loss 等高线 + 各优化器轨迹
xx, yy = torch.meshgrid(torch.linspace(-1, 4, 200), torch.linspace(-1, 3, 200), indexing='xy')
zz = beale(xx, yy)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, torch.log10(zz + 1), levels=20, alpha=0.7)
plt.plot(3, 0.5, 'k*', ms=14, label='global min')
for name, t in trajs.items():
    plt.plot(t[:, 0], t[:, 1], '-o', ms=2, lw=1, label=name)
plt.legend(loc='upper right', fontsize=8); plt.xlabel('x'); plt.ylabel('y')
plt.title('optimizer trajectories on log10(Beale + 1)'); plt.show()

print('--- final f(x, y) ---')
for name, t in trajs.items():
    fx = beale(t[-1, 0], t[-1, 1]).item()
    print(f'  {name:25s}  final={t[-1].tolist()}  f={fx:.4f}')

**观察**：

- **SGD+momentum**、**RMSprop**、**Adam** 都成功逼近全局最小附近。Adam 需要比 SGD 大一个量级的 lr——这与 Adam 内部对梯度做 RMS 归一化的特性吻合。
- **纯 SGD** 步长太小、进度最慢。
- 这只是一个二维示例，结论方向：自适应优化器在病态曲率（如 Beale 函数细长的山谷）上有优势；momentum 在大多数 loss 面上有用。

## 2. mini-batch 大小对收敛速度的影响

用 LeNet 在 MNIST 子集上，固定优化器、固定 lr，只改 batch_size。

In [ ]:
import os
from torch.utils.data import Subset
from torchvision import datasets, transforms

CACHE = os.path.expanduser('~/.cache/torch_data')
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST(CACHE, train=True, download=True, transform=tfm)
test_full  = datasets.MNIST(CACHE, train=False, download=True, transform=tfm)

torch.manual_seed(0)
tr_idx = torch.randperm(len(train_full))[:5000].tolist()
te_idx = torch.randperm(len(test_full))[:1000].tolist()
train_set = Subset(train_full, tr_idx)
test_set  = Subset(test_full,  te_idx)
test_loader = DataLoader(test_set, batch_size=256)

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.flatten(1)
        x = F.relu(self.fc1(x)); x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
def train_with_bs(bs, epochs=3, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    loader = DataLoader(train_set, batch_size=bs, shuffle=True)
    model = LeNet5(); opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    accs = []
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            correct = total = 0
            for x, y in test_loader:
                correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
        accs.append(correct / total)
    return accs

bs_accs = {bs: train_with_bs(bs) for bs in [16, 64, 256]}
for bs, accs in bs_accs.items():
    print(f'bs={bs:4d}  test_acc by epoch: ' + ', '.join(f'{a:.3f}' for a in accs))

小 batch 每个 epoch 走更多步（同样数据），所以收敛**早**——但 batch 太小时单步梯度方差大，最终精度不一定更好。Practical 选择常是 32-256。

## 3. 参数初始化：Xavier vs Kaiming

目标：信号通过深网络后，每层的方差应当大致保持稳定（既不爆炸也不消失）。

- **Xavier (`xavier_normal_`)**: 假设 tanh / sigmoid 这样的近线性激活，方差按 $2 / (n_\text{in} + n_\text{out})$ 缩放。
- **Kaiming (`kaiming_normal_`)**: 假设 ReLU 类激活（一半被砍掉），方差按 $2 / n_\text{in}$ 缩放。

下面构造一个 10 层的全连接网络，看不同初始化下每层输出的标准差。

In [ ]:
def build_deep_mlp(init_kind, n_layers=10, dim=512, activation='relu'):
    torch.manual_seed(0)
    layers = []
    for _ in range(n_layers):
        layers.append(nn.Linear(dim, dim))
        layers.append(nn.ReLU() if activation == 'relu' else nn.Tanh())
    model = nn.Sequential(*layers)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if init_kind == 'naive':
                nn.init.normal_(m.weight, std=0.05)
            elif init_kind == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_kind == 'kaiming':
                nn.init.kaiming_normal_(m.weight, nonlinearity=activation)
            nn.init.zeros_(m.bias)
    return model

def collect_std(model):
    """Forward 一次，收集每个 Linear 层输出的标准差。"""
    x = torch.randn(64, 512)
    stds = []
    with torch.no_grad():
        for m in model:
            x = m(x)
            if isinstance(m, nn.Linear):
                stds.append(x.std().item())
    return stds

for kind in ['naive', 'xavier', 'kaiming']:
    stds = collect_std(build_deep_mlp(kind, activation='relu'))
    print(f'{kind:8s} (ReLU):  ' + ', '.join(f'{s:7.3f}' for s in stds))

**naive 高斯初始化** 让输出方差按 ~10x 速度衰减；**Xavier** 偏小（针对 tanh 设计）；**Kaiming** 在 ReLU 下能保持每层方差在同一量级。这就是深层 ReLU 网络默认用 Kaiming 的原因（PyTorch `nn.Linear` 的默认就是 Kaiming uniform）。

## 4. BatchNorm 的训练加速效果

BN 在 batch 维度上把每个特征通道**归一化**后再线性变换。它能让我们使用更大的学习率、加快收敛，并自带一些正则化效果。

In [ ]:
class MLPNoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128), nn.ReLU(),
            nn.Linear(128,    64), nn.ReLU(),
            nn.Linear(64,     10),
        )
    def forward(self, x): return self.net(x)

class MLPWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128,    64), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64,     10),
        )
    def forward(self, x): return self.net(x)

def train_mlp(model, epochs=3, lr=1e-1, bs=64):
    torch.manual_seed(0)
    loader = DataLoader(train_set, batch_size=bs, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=lr)
    losses = []
    for _ in range(epochs):
        model.train()
        ep_running, n = 0., 0
        for x, y in loader:
            opt.zero_grad(); loss = F.cross_entropy(model(x), y); loss.backward(); opt.step()
            ep_running += loss.item() * x.size(0); n += x.size(0)
        losses.append(ep_running / n)
    return losses

no_bn = train_mlp(MLPNoBN())
with_bn = train_mlp(MLPWithBN())
for ep, (a, b) in enumerate(zip(no_bn, with_bn), 1):
    print(f'epoch {ep}: no-BN train_loss={a:.4f}   with-BN={b:.4f}')

在大学习率（`lr=0.1`）下，无 BN 的网络收敛慢甚至发散；BN 让它稳稳下降。这是 BN 最常被引用的好处。

## 5. Dropout 的正则化效果

在一个会过拟合的小数据 / 大网络组合上对比。

In [ ]:
# 小数据：MNIST 1000 训练样本
tiny_train = Subset(train_full, torch.randperm(len(train_full))[:1000].tolist())
tiny_train_loader = DataLoader(tiny_train, batch_size=64, shuffle=True)

class BigMLP(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,   256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,    10),
        )
    def forward(self, x): return self.net(x)

def train_and_eval(model, epochs=15, lr=1e-2):
    torch.manual_seed(0)
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    train_acc, test_acc = [], []
    for _ in range(epochs):
        model.train()
        for x, y in tiny_train_loader:
            opt.zero_grad(); F.cross_entropy(model(x), y).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            c = t = 0
            for x, y in tiny_train_loader:
                c += (model(x).argmax(1) == y).sum().item(); t += y.size(0)
            train_acc.append(c / t)
            c = t = 0
            for x, y in test_loader:
                c += (model(x).argmax(1) == y).sum().item(); t += y.size(0)
            test_acc.append(c / t)
    return train_acc, test_acc

tr_no, te_no = train_and_eval(BigMLP(dropout=0.0))
tr_dp, te_dp = train_and_eval(BigMLP(dropout=0.5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tr_no, label='train (no dropout)'); axes[0].plot(te_no, label='test (no dropout)')
axes[0].plot(tr_dp, label='train (dropout 0.5)'); axes[0].plot(te_dp, label='test (dropout 0.5)')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('acc'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([a - b for a, b in zip(tr_no, te_no)], label='train-test gap (no dropout)')
axes[1].plot([a - b for a, b in zip(tr_dp, te_dp)], label='train-test gap (dropout 0.5)')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('train acc - test acc'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'no dropout : final train acc={tr_no[-1]:.3f}  test={te_no[-1]:.3f}  gap={tr_no[-1]-te_no[-1]:+.3f}')
print(f'dropout 0.5: final train acc={tr_dp[-1]:.3f}  test={te_dp[-1]:.3f}  gap={tr_dp[-1]-te_dp[-1]:+.3f}')

Dropout 让训练-测试准确率差距（generalization gap）变小，是经典正则化手段。注意 `model.eval()` 会自动关掉 dropout，所以 `runner.evaluate()` 不需要手动处理。

## 6. 学习率调度

`torch.optim.lr_scheduler` 提供了一系列调度器。常用：

- **`StepLR(gamma=0.1, step_size=30)`**：每 30 epoch 把 lr × 0.1
- **`CosineAnnealingLR(T_max=...)`**：余弦衰减（现代默认）
- **`ReduceLROnPlateau`**：dev 指标停止改进时降 lr

下面画三种调度器在 100 epoch 上的 lr 曲线。

In [ ]:
model = nn.Linear(2, 2)
epochs = 100
schedulers = {
    'StepLR(30, 0.5)':            lambda: optim.lr_scheduler.StepLR(optim.SGD(model.parameters(), lr=0.1), step_size=30, gamma=0.5),
    'CosineAnnealingLR(100)':     lambda: optim.lr_scheduler.CosineAnnealingLR(optim.SGD(model.parameters(), lr=0.1), T_max=epochs),
    'ExponentialLR(0.97)':        lambda: optim.lr_scheduler.ExponentialLR(optim.SGD(model.parameters(), lr=0.1), gamma=0.97),
}

plt.figure(figsize=(9, 4))
for name, factory in schedulers.items():
    sched = factory()
    opt_ref = sched.optimizer
    lrs = [opt_ref.param_groups[0]['lr']]
    for _ in range(epochs):
        opt_ref.step()                    # 实际训练中这一步是真正用 lr 的地方
        sched.step()
        lrs.append(opt_ref.param_groups[0]['lr'])
    plt.plot(lrs, label=name)
plt.xlabel('epoch'); plt.ylabel('learning rate'); plt.legend(); plt.grid(alpha=0.3); plt.show()

**典型使用模板**：

```python
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
for epoch in range(epochs):
    train_one_epoch(model, optimizer, ...)
    scheduler.step()                # 注意：scheduler.step() 一般 *per-epoch*
```

## 小结

- **优化器**：Adam 是无脑默认；SGD+Momentum 在 CV 大数据上微调常超过 Adam。
- **batch size**：32-256 是甜区。太小方差大，太大每 epoch 步数太少。
- **初始化**：用 `nn.init.kaiming_normal_/uniform_` 配 ReLU；`xavier_*` 配 tanh/sigmoid。PyTorch `nn.Linear/Conv2d` 默认就用 Kaiming uniform——大多数情况下不用手动设。
- **BatchNorm**：现代 CNN 标配；让大学习率成为可能。注意 `train()` / `eval()` 模式切换。
- **Dropout**：过拟合时祭出，0.2-0.5 之间挑。`eval()` 自动关掉。
- **lr_scheduler**：CosineAnnealingLR / OneCycleLR 是现代默认；`scheduler.step()` 通常每 epoch 调一次。
- **weight decay**：直接挂在优化器上 (`weight_decay=...`)，等价于 L2 正则。CV 常用 5e-4，Transformer 常用 0.1。